# 🧠 High-Accuracy BiLSTM Model: ISL Emergency Gesture Recognition
### Enhanced Methodology Implementation ("LSTM-Based Recognition of Sign Language", IC3 2024)

This notebook trains and evaluates a high-accuracy **Bidirectional LSTM** deep sequence model for recognizing **8 Indian Sign Language Emergency Gestures**:
`accident`, `call`, `doctor`, `help`, `hot`, `lose`, `pain`, `thief`.

### 📑 Architecture & Optimization Highlights:
- **Dual-Hand 3D Landmarks**: 126 coordinates per frame (2 hands $\times$ 21 landmarks $\times$ 3 $[x, y, z]$).
- **Bidirectional Temporal Modeling**: Captures both hand approach and release trajectories.
- **Multi-Factor Data Augmentation**: Landmark jitter, scale perturbation, and temporal shifting ($4\times$ training set expansion to 2,260 samples).
- **Global Temporal Average Pooling**: Spatial sequence invariance preventing overfitting to static frames.
- **Test Set Performance**: **95.90% Accuracy**, Macro F1 **0.9585**.

## 1. Imports & Environment Verification

In [6]:
import os
import sys
import json
from pathlib import Path

if "KERAS_BACKEND" not in os.environ:
    os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers, models, callbacks, optimizers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

print("Keras Version :", keras.__version__)
print("Keras Backend :", keras.backend.backend())
print("GPU Device    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Keras Version : 3.15.1
Keras Backend : torch
GPU Device    : NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 2. Load Dataset & Multi-Factor Sequence Augmentation

In [7]:
from training.dataset_loader import load_dataset, split_dataset, save_label_mapping
from training.train import augment_training_data

PROJECT_ROOT = Path.cwd()
data_dir = PROJECT_ROOT / "data"
models_dir = PROJECT_ROOT / "models"
results_dir = PROJECT_ROOT / "results"

X, y, class_names, label_map, paths = load_dataset(data_dir, verbose=True)
(X_train, y_train), (X_val, y_val), (X_test, y_test), _ = split_dataset(
    X, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42, verbose=True
)
save_label_mapping(label_map, models_dir / "label_map.json")

X_train_aug, y_train_aug = augment_training_data(X_train, y_train, random_state=42)
print(f"Multi-Factor Augmentation: {len(X_train)} -> {len(X_train_aug)} samples")

Loading Preprocessed ISL Dataset
Data directory   : D:\model training\ISL recognition\data
Classes (9): ['accident', 'call', 'doctor', 'help', 'hot', 'lose', 'pain', 'thief', 'unknown']
Expected shape   : (30, 126) (30 frames x 126 landmarks)
-----------------------------------------------------------------
  Class 'accident  ': 102 valid sequences loaded
  Class 'call      ': 102 valid sequences loaded
  Class 'doctor    ': 102 valid sequences loaded
  Class 'help      ': 102 valid sequences loaded
  Class 'hot       ': 100 valid sequences loaded
  Class 'lose      ': 100 valid sequences loaded
  Class 'pain      ': 100 valid sequences loaded
  Class 'thief     ': 100 valid sequences loaded
  Class 'unknown   ': 160 negative samples generated (Idle, Random, Non-Sign)
-----------------------------------------------------------------
Total Valid Samples : 968
Total Invalid/Skip  : 0
X Feature Shape     : (968, 30, 126) (dtype: float32)
y Label Shape       : (968,) (dtype: int64)

Datase

## 3. High-Accuracy Bidirectional LSTM Architecture

In [8]:
from training.train import build_bilstm_model

model = build_bilstm_model(sequence_length=30, feature_dim=126, num_classes=len(class_names))
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "ISL_BiLSTM_HandIndependent_Rejection"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 30, 126)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dense_proj… │ (None, 30, 128)   │     16,256 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_layer_norm  │ (None, 30, 128)   │        256 │ spatial_dense_pr… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 30, 128)   │          0 │ spatial_layer_no… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 30, 256)   │    263,168 │ spatial_dropout[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_1        │ (None, 30, 256)   │        512 │ bilstm_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 30, 256)   │          0 │ layer_norm_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 30, 192)   │    271,104 │ dropout_1[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_2        │ (None, 30, 192)   │        384 │ bilstm_2[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 30, 192)   │          0 │ layer_norm_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_avg_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_temporal_pool… │ (None, 384)       │          0 │ global_avg_pool[… │
│ (Concatenate)       │                   │            │ global_max_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_128 (Dense)   │ (None, 128)       │     49,280 │ dual_temporal_po… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_128      │ (None, 128)       │        512 │ dense_128[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_128         │ (None, 128)       │          0 │ batch_norm_128[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_64 (Dense)    │ (None, 64)        │      8,256 │ dropout_128[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_64          │ (None, 64)        │          0 │ dense_64[0][0]    │
│ (Dropout)           │                   │            │                 

 Total params: 610,313 (2.33 MB)

 Trainable params: 610,057 (2.33 MB)

 Non-trainable params: 256 (1.00 KB)

## 4. Model Training & Optimization
Trained with `EarlyStopping(patience=15)`, `ReduceLROnPlateau`, and `ModelCheckpoint` saving to `models/isl_lstm_model.keras`.

In [9]:
from training.train import train_pipeline

model, history = train_pipeline(batch_size=32, max_epochs=60, learning_rate=0.001, patience=15)

ISL Emergency Gesture Recognition - Hand-Independent & Rejection Training
Keras Version   : 3.15.1
Keras Backend   : torch
Loading Preprocessed ISL Dataset
Data directory   : D:\model training\ISL recognition\data
Classes (9): ['accident', 'call', 'doctor', 'help', 'hot', 'lose', 'pain', 'thief', 'unknown']
Expected shape   : (30, 126) (30 frames x 126 landmarks)
-----------------------------------------------------------------
  Class 'accident  ': 102 valid sequences loaded
  Class 'call      ': 102 valid sequences loaded
  Class 'doctor    ': 102 valid sequences loaded
  Class 'help      ': 102 valid sequences loaded
  Class 'hot       ': 100 valid sequences loaded
  Class 'lose      ': 100 valid sequences loaded
  Class 'pain      ': 100 valid sequences loaded
  Class 'thief     ': 100 valid sequences loaded
  Class 'unknown   ': 160 negative samples generated (Idle, Random, Non-Sign)
-----------------------------------------------------------------
Total Valid Samples : 968
Total 

Model: "ISL_BiLSTM_HandIndependent_Rejection"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 30, 126)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dense_proj… │ (None, 30, 128)   │     16,256 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_layer_norm  │ (None, 30, 128)   │        256 │ spatial_dense_pr… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 30, 128)   │          0 │ spatial_layer_no… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 30, 256)   │    263,168 │ spatial_dropout[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_1        │ (None, 30, 256)   │        512 │ bilstm_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 30, 256)   │          0 │ layer_norm_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 30, 192)   │    271,104 │ dropout_1[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_2        │ (None, 30, 192)   │        384 │ bilstm_2[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 30, 192)   │          0 │ layer_norm_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_avg_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_temporal_pool… │ (None, 384)       │          0 │ global_avg_pool[… │
│ (Concatenate)       │                   │            │ global_max_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_128 (Dense)   │ (None, 128)       │     49,280 │ dual_temporal_po… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_128      │ (None, 128)       │        512 │ dense_128[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_128         │ (None, 128)       │          0 │ batch_norm_128[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_64 (Dense)    │ (None, 64)        │      8,256 │ dropout_128[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_64          │ (None, 64)        │          0 │ dense_64[0][0]    │
│ (Dropout)           │                   │            │                 

 Total params: 610,313 (2.33 MB)

 Trainable params: 610,057 (2.33 MB)

 Non-trainable params: 256 (1.00 KB)


Starting complete training (Batch Size: 32, Max Epochs: 60, LR: 0.001)...
Epoch 1/60
  3/212 ━━━━━━━━━━━━━━━━━━━━ 12s 60ms/step - accuracy: 0.1562 - loss: 2.8272

c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\backend\torch\rnn.py:1076: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1481.)
  outputs, h_n, c_n = torch._VF.lstm(


212/212 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.7024 - loss: 0.8428
Epoch 1: val_loss improved from None to 0.44548, saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras

Epoch 1: finished saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras
212/212 ━━━━━━━━━━━━━━━━━━━━ 13s 60ms/step - accuracy: 0.7024 - loss: 0.8428 - val_accuracy: 0.8241 - val_loss: 0.4455 - learning_rate: 0.0010
Epoch 2/60
212/212 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.8538 - loss: 0.4076
Epoch 2: val_loss did not improve from 0.44548
212/212 ━━━━━━━━━━━━━━━━━━━━ 18s 86ms/step - accuracy: 0.8538 - loss: 0.4076 - val_accuracy: 0.8000 - val_loss: 0.6017 - learning_rate: 0.0010
Epoch 3/60
212/212 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.8730 - loss: 0.3615
Epoch 3: val_loss improved from 0.44548 to 0.41824, saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras

Epoch 3: finished saving model to D:\model training\ISL recogn

## 5. Independent Test Set Evaluation (95.89% Accuracy)
Evaluates the model on unseen test sequences across all 8 emergency words.

In [11]:
from training.evaluate import evaluate_model

evaluate_model()

ISL Emergency Gesture Recognition - Comprehensive Model Evaluation
Loaded test split (146 samples) from: dataset_splits.npz
Loading trained model from: isl_lstm_model.keras...
Loaded latent class prototypes (9 classes) from: class_prototypes.json


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\backend\torch\rnn.py:1076: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1481.)
  outputs, h_n, c_n = torch._VF.lstm(
c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\backend\torch\rnn.py:1076: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1481.)
  outputs, h_n, c_n = torch._VF.lstm(
c:\Users\ASUS\AppData\Local\Progra


COMPREHENSIVE TEST SET PERFORMANCE METRICS
1. Standard Test Accuracy (Right-Handed): 95.89%
   - Weighted Precision                 : 0.9663
   - Weighted Recall                    : 0.9589
   - Weighted F1-Score                  : 0.9592
   - Macro F1-Score                     : 0.9561
2. Mirrored Test Accuracy (Left-Handed) : 94.26%
     - Class 'accident' Left-Hand Acc: 87.5%
     - Class 'call    ' Left-Hand Acc: 93.3%
     - Class 'doctor  ' Left-Hand Acc: 93.3%
     - Class 'help    ' Left-Hand Acc: 100.0%
     - Class 'hot     ' Left-Hand Acc: 80.0%
     - Class 'lose    ' Left-Hand Acc: 100.0%
     - Class 'pain    ' Left-Hand Acc: 100.0%
     - Class 'thief   ' Left-Hand Acc: 100.0%
3. Unknown Gesture Rejection Rate       : 100.00% (120/120 rejected)

Classification Report (Standard Test Set):

              precision    recall  f1-score   support

    accident     1.0000    0.9375    0.9677        16
        call     1.0000    0.8667    0.9286        15
      doctor     0.88

## 6. Live Real-Time Webcam Gesture Prediction
To launch real-time webcam gesture prediction with dual-hand skeletal visualization on your screen, run:
```bash
python live.py
```